# Notebook 00: Setup and data acquisition
**Project:** Cyclone risk to Odisha's transmission towers and a design-standard review (CDRI)

**What this notebook does**
1. Mounts Google Drive and creates the project folders
2. Loads the Odisha transmission network (41,814 towers, 227 substations, 327 lines) and the 128 Fani-damaged towers from Raj et al.'s public GitHub repo
3. Downloads the coastline and district boundaries, then tags every tower with district, distance to coast and coastal band (0–20 / 20–60 / >60 km)
4. Unpacks the STORM synthetic tracks (North Indian basin, 10,000 years) and keeps the storms that pass within 300 km of the Odisha coast
5. Downloads IBTrACS (North Indian basin) and extracts Fani plus the validation cyclones
6. Downloads hourly station winds for Fani (NOAA ISD, via Meteostat)
7. Draws a QA map and writes summary tables and a data inventory

**Before you run:** upload the STORM zip (`STORM IBTrACS present climate synthetic tropical cyclone tracks_4_all.zip`) to `MyDrive/CDRI_Odisha/data/raw/`.

Everything is free and open: Colab, GitHub, Natural Earth, GADM / geoBoundaries, STORM (4TU), IBTrACS (NOAA), Meteostat.

Run the cells top to bottom (Runtime → Run all). Each section can be re-run on its own because finished downloads are reused.

## 0. Environment and folders

In [ ]:
import sys, subprocess, importlib

def ensure(pkg, import_name=None):
    """Install a package only if it is missing (Colab already has most)."""
    name = import_name or pkg.split('[')[0].split('<')[0].split('>')[0].split('=')[0]
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

for p, mod in [('geopandas', None), ('pyarrow', None), ('openpyxl', None), ('meteostat>=1.6,<2', 'meteostat')]:
    ensure(p, mod)

In [ ]:
import os, io, re, json, math, zipfile, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.spatial import cKDTree
from shapely.geometry import Point, LineString, MultiLineString, box
import requests

warnings.filterwarnings('ignore', category=UserWarning)

def union(gs):
    """Dissolve a GeoSeries/GeoDataFrame to one geometry (works on old and new geopandas)."""
    return gs.union_all() if hasattr(gs, 'union_all') else gs.unary_union
pd.set_option('display.width', 160)

IN_COLAB = 'google.colab' in sys.modules
GEE_PROJECT = 'goggle-earth-499404'   # your Google Earth Engine Cloud project (used from Notebook 01)
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
        ROOT = Path('/content/drive/MyDrive/CDRI_Odisha')
    except Exception as e:
        print('Drive mount failed:', e)
        print('To fix: Runtime → Disconnect and delete runtime, run this cell again, sign in with the SAME Google\n'
              'account you use for Colab, and tick ALL the permission boxes on the consent screen.\n'
              'Continuing in TEMPORARY Colab storage for now (deleted when the runtime stops).')
        ROOT = Path('/content/CDRI_Odisha')
else:
    ROOT = Path(os.environ.get('CDRI_ROOT', Path.cwd() / 'CDRI_Odisha'))

DIRS = {k: ROOT / v for k, v in {
    'raw': 'data/raw', 'external': 'data/external', 'interim': 'data/interim',
    'processed': 'data/processed', 'figures': 'outputs/figures', 'tables': 'outputs/tables'}.items()}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)

INVENTORY = []   # one row per dataset, written out at the end
def log_data(name, source, path, n=None, status='ok', note=''):
    INVENTORY.append(dict(dataset=name, source=source, local_path=str(path), records=n, status=status,
                          note=note, accessed_utc=datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')))

def download(url, dest, timeout=120, min_bytes=1000):
    """Download once; reuse the file if it already exists. Returns the path, or None on failure."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > min_bytes:
        return dest
    try:
        r = requests.get(url, timeout=timeout, headers={'User-Agent': 'CDRI-Odisha-research/0.1'}, stream=True)
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(dest)
        return dest
    except Exception as e:
        print(f'  ! download failed: {url}\n    {type(e).__name__}: {e}')
        return None

## 1. Study settings
- **Study domain:** the coastal districts plus Khordha and Cuttack, within 60 km of the coast (the CDRI screening band), split 0–20 km and 20–60 km.
- **Fani zone:** 50 km around the Puri landfall.
- **Projected CRS:** UTM 45N (EPSG:32645), which covers the Odisha coast; used for all distances.

In [ ]:
CRS_GEO, CRS_M = 'EPSG:4326', 'EPSG:32645'
BBOX_ODISHA = (81.3, 17.8, 87.6, 22.6)            # lon_min, lat_min, lon_max, lat_max
BBOX_COAST  = (84.0, 17.5, 88.2, 22.4)            # for clipping the coastline (incl. neighbouring AP / WB coast)
FANI_LANDFALL = (85.82, 19.80)                    # lon, lat (approx., near Puri); refined from IBTrACS in §6
FANI_RADIUS_KM = 50
TRACK_BUFFER_KM = 300                             # keep synthetic storms passing within this distance of the coast
BANDS = [(0, 20, '0-20 km'), (20, 60, '20-60 km'), (60, 1e9, '>60 km')]

# District aliases (spellings differ between GADM, geoBoundaries and Census)
STUDY_DISTRICTS = {
    'Balasore':      ['balasore', 'baleshwar', 'baleswar'],
    'Bhadrak':       ['bhadrak'],
    'Kendrapara':    ['kendrapara', 'kendrapada'],
    'Jagatsinghpur': ['jagatsinghpur', 'jagatsinghapur', 'jagatsingpur'],
    'Puri':          ['puri'],
    'Ganjam':        ['ganjam'],
    'Khordha':       ['khordha', 'khurda', 'khorda'],
    'Cuttack':       ['cuttack'],
}

# WMO/TD-1555 Table 1.1 gust factors G(3 s, To) at +10 m, tropical cyclones (used in later notebooks)
WMO_GUST_3S = pd.DataFrame(
    {'To_60s': [1.49, 1.36, 1.23, 1.11], 'To_180s': [1.58, 1.44, 1.31, 1.17], 'To_600s': [1.66, 1.52, 1.38, 1.23]},
    index=['in-land', 'off-land', 'off-sea', 'at-sea'])
WMO_GUST_3S.to_csv(DIRS['tables'] / 'wmo_td1555_gust_factors_3s.csv')
WMO_GUST_3S

## 2. Transmission network and Fani damage (Raj, Kumar & Bhatia, IIT Gandhinagar)
Public repo: `surendervraj/cyclone-prepardness-for-regional-power-transmission-network`. It holds
- `Network_Node.mat`: 42,035 nodes [id, lat, lon, is_substation, line_id], of which 41,814 are towers
- `Network_Edge.mat`: tower-to-tower links (so we can rebuild line geometry)
- `Network_details.xlsx`: 227 substations (names, population weight, centrality) and 327 lines with voltage
- `Step2 -Damage data_fragility/*.csv`: the 128 Fani-damaged towers from OPTCL (obtained through RTI): 87 collapsed, 23 cross-arm, 18 other

Please cite Raj et al. when you use these data.

In [ ]:
REPO_URL = 'https://github.com/surendervraj/cyclone-prepardness-for-regional-power-transmission-network'
REPO_DIR = DIRS['external'] / 'raj_network'
if not (REPO_DIR / '.git').exists():
    src = os.environ.get('CDRI_REPO_SRC')                     # optional local copy (offline testing)
    if src:
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

BASE = REPO_DIR / 'Base case _ no strengthening_with FII index'
DMG  = REPO_DIR / 'Step2 -Damage data_fragility'

node = loadmat(BASE / 'Network_Node.mat')['Network_Node']
link = loadmat(BASE / 'Network_Edge.mat')['Network_Edge'].astype(int)
xls  = pd.read_excel(BASE / 'Network_details.xlsx', sheet_name=None)

nodes = pd.DataFrame(node, columns=['node_id', 'lat', 'lon', 'is_sub', 'line_id'])
nodes[['node_id', 'is_sub', 'line_id']] = nodes[['node_id', 'is_sub', 'line_id']].astype(int)

subs = xls['Nodes'].rename(columns={'Node Id': 'sub_id', 'Substation Name': 'name', 'Latitude': 'lat',
                                    'Longitude': 'lon', 'Population weight': 'pop_weight', 'Degree': 'degree',
                                    'Betweeness': 'betweenness', 'Closeness': 'closeness', 'Eigen': 'eigen'})
subs['name'] = subs['name'].astype(str).str.strip()
lines = xls['Edges'].rename(columns={'Edge Id': 'line_id', 'Start Node': 'from_sub', 'End Node': 'to_sub',
                                     'Voltage(V)': 'voltage_kv'})   # the column says "V" but the values are kV
name_of = subs.set_index('sub_id')['name']
lines['from_name'] = lines['from_sub'].map(name_of)
lines['to_name']   = lines['to_sub'].map(name_of)
lines['line_name'] = lines['from_name'] + ' – ' + lines['to_name']

towers = nodes[nodes.is_sub == 0].drop(columns='is_sub').rename(columns={'node_id': 'tower_id'})
towers = towers.merge(lines[['line_id', 'voltage_kv', 'line_name', 'from_sub', 'to_sub']], on='line_id', how='left')

print(f'Towers: {len(towers):,} | substations: {len(subs)} | lines: {len(lines)}')
print(towers.voltage_kv.value_counts().sort_index().rename('towers by kV').to_string())
assert len(towers) == 41814, 'Unexpected tower count: check the repo version'

In [ ]:
# Fani damage (OPTCL records via Raj et al.)
dmg_files = {'collapse': 'Collapsed Towers.csv', 'crossarm': 'Cross arm damaged Towers.csv', 'partial_other': 'Other damage Towers.csv'}
dmg = []
for state, f in dmg_files.items():
    d = pd.read_csv(DMG / f)
    d['damage_state'] = state
    dmg.append(d)
dmg = pd.concat(dmg, ignore_index=True)
dmg.columns = [c.strip() for c in dmg.columns]
dmg = dmg.rename(columns={'Line Name': 'optcl_line_name', 'Line Id': 'optcl_line_id', 'Location Number': 'location_no',
                          'Nodenumber in network': 'tower_id', 'Latitude': 'lat_dmg', 'Longitude': 'lon_dmg',
                          'Damage Observed': 'damage_observed', 'Remarks': 'remarks'})
dmg['optcl_line_name'] = dmg['optcl_line_name'].str.strip()
dmg['optcl_kv'] = dmg['optcl_line_id'].str.split('_').str[0].astype(int)
print(dmg.damage_state.value_counts().to_string(), '\nTotal damaged towers:', len(dmg))

# Check that the damage records point at real towers
chk = dmg.merge(towers[['tower_id', 'lat', 'lon', 'voltage_kv']], on='tower_id', how='left')
offset_m = np.hypot((chk.lat - chk.lat_dmg) * 111_000, (chk.lon - chk.lon_dmg) * 111_000 * np.cos(np.radians(chk.lat)))
print(f'Tower id found for {chk.lat.notna().sum()}/{len(chk)} records; median offset to network tower {np.nanmedian(offset_m):.0f} m')
print('Voltage agreement (OPTCL id vs network):', (chk.optcl_kv == chk.voltage_kv).mean().round(2))

# One row per tower (keep the most severe state if a tower appears twice)
sev = {'collapse': 3, 'crossarm': 2, 'partial_other': 1}
dmg_t = (dmg.assign(sev=dmg.damage_state.map(sev)).sort_values('sev', ascending=False)
            .drop_duplicates('tower_id')[['tower_id', 'damage_state', 'optcl_line_name', 'damage_observed']])
towers = towers.merge(dmg_t, on='tower_id', how='left')
towers['damaged'] = towers['damage_state'].notna()
towers['collapsed'] = towers['damage_state'].eq('collapse')
print('\nDamaged towers by OPTCL line:')
print(dmg.groupby(['optcl_line_name', 'optcl_kv']).size().sort_values(ascending=False).to_string())
log_data('Raj et al. Odisha transmission network', REPO_URL, REPO_DIR, len(towers))
log_data('Fani tower damage (OPTCL via RTI, Raj et al.)', REPO_URL, DMG, len(dmg))

In [ ]:
# Rebuild line geometry from the tower-to-tower links
xy = nodes.set_index('node_id')[['lon', 'lat']]
lk = pd.DataFrame(link, columns=['link_id', 'a', 'b'])
lk = lk[lk.a.isin(xy.index) & lk.b.isin(xy.index)]
lk['line_id'] = nodes.set_index('node_id').line_id.reindex(lk.b).values
lk.loc[lk.line_id == 0, 'line_id'] = nodes.set_index('node_id').line_id.reindex(lk.loc[lk.line_id == 0, 'a']).values
A, B = xy.reindex(lk.a).values, xy.reindex(lk.b).values
lk['geometry'] = [LineString([tuple(p), tuple(q)]) for p, q in zip(A, B)]
line_geom = (gpd.GeoDataFrame(lk, geometry='geometry', crs=CRS_GEO).dissolve('line_id').reset_index()[['line_id', 'geometry']])
gdf_lines = gpd.GeoDataFrame(lines.merge(line_geom, on='line_id', how='left'), geometry='geometry', crs=CRS_GEO)
gdf_lines['length_km'] = gdf_lines.to_crs(CRS_M).length / 1000
print(f'Line geometry rebuilt for {gdf_lines.geometry.notna().sum()} of {len(gdf_lines)} lines; '
      f'total route length ≈ {gdf_lines.length_km.sum():,.0f} km')

gdf_towers = gpd.GeoDataFrame(towers, geometry=gpd.points_from_xy(towers.lon, towers.lat), crs=CRS_GEO)
gdf_subs = gpd.GeoDataFrame(subs, geometry=gpd.points_from_xy(subs.lon, subs.lat), crs=CRS_GEO)

# Approximate span: distance to the next tower on the same line (useful later for the CDRI span indicator)
lk_m = gpd.GeoSeries(lk.geometry, crs=CRS_GEO).to_crs(CRS_M).length
span = pd.Series(lk_m.values, index=lk.b.values).groupby(level=0).max()
gdf_towers['span_m'] = gdf_towers.tower_id.map(span)
print(gdf_towers.groupby('voltage_kv').span_m.median().round(0).rename('median span (m)').to_string())

## 3. Coastline, distance to coast and coastal bands
Primary source: Natural Earth 10 m coastline (public domain). If the download fails, a simplified Odisha coast polyline is used, which is enough for 20/60 km bands but should be replaced before final maps.

In [ ]:
NE_URL = 'https://naciscdn.org/naturalearth/10m/physical/ne_10m_coastline.zip'
ne_zip = download(NE_URL, DIRS['raw'] / 'ne_10m_coastline.zip')
coast = None
if ne_zip:
    try:
        coast = gpd.read_file(f'zip://{ne_zip}').to_crs(CRS_GEO)
        coast = gpd.clip(coast, box(*BBOX_COAST))
        coast_src = 'Natural Earth 10 m'
    except Exception as e:
        print('  ! could not read Natural Earth coastline:', e)
if coast is None or coast.empty:
    # Fallback: simplified Odisha coast (approximate points, south to north)
    pts = [(84.70, 19.07), (84.92, 19.26), (85.12, 19.45), (85.40, 19.62), (85.62, 19.72), (85.83, 19.80),
           (86.10, 19.88), (86.40, 20.05), (86.67, 20.26), (86.80, 20.50), (86.96, 20.80), (86.92, 21.10),
           (87.05, 21.45), (87.45, 21.60), (87.80, 21.70)]
    coast = gpd.GeoDataFrame(geometry=[LineString(pts)], crs=CRS_GEO)
    coast_src = 'FALLBACK simplified polyline (replace before final maps)'
print('Coastline source:', coast_src)
log_data('Coastline', NE_URL, ne_zip or 'fallback', len(coast), 'ok' if 'Natural' in coast_src else 'fallback')

coast_m = union(coast.to_crs(CRS_M))
twr_m = gdf_towers.to_crs(CRS_M)
gdf_towers['dist_coast_km'] = twr_m.distance(coast_m).values / 1000
gdf_subs['dist_coast_km'] = gdf_subs.to_crs(CRS_M).distance(coast_m).values / 1000
def band_of(d):
    for lo, hi, lab in BANDS:
        if lo <= d < hi:
            return lab
gdf_towers['band'] = gdf_towers.dist_coast_km.apply(band_of)
gdf_subs['band'] = gdf_subs.dist_coast_km.apply(band_of)

fani_pt = gpd.GeoSeries([Point(FANI_LANDFALL)], crs=CRS_GEO).to_crs(CRS_M).iloc[0]
gdf_towers['dist_fani_landfall_km'] = twr_m.distance(fani_pt).values / 1000
gdf_towers['in_fani_zone'] = gdf_towers.dist_fani_landfall_km <= FANI_RADIUS_KM
print(gdf_towers.band.value_counts().reindex([b[2] for b in BANDS]).rename('towers per band').to_string())
print(f'Towers within {FANI_RADIUS_KM} km of the Fani landfall: {gdf_towers.in_fani_zone.sum():,}; '
      f'damaged towers inside that radius: {(gdf_towers.in_fani_zone & gdf_towers.damaged).sum()} of {gdf_towers.damaged.sum()}')

## 4. District boundaries
GADM 4.1 (free for non-commercial and academic use). Falls back to geoBoundaries (CC BY 4.0).

In [ ]:
def norm(s):
    return re.sub(r'[^a-z]', '', str(s).lower())

districts = None
gadm_zip = download('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IND_2.json.zip', DIRS['raw'] / 'gadm41_IND_2.json.zip', timeout=300)
if gadm_zip:
    try:
        g = gpd.read_file(f'zip://{gadm_zip}')
        districts = g[g.NAME_1.map(norm).isin(['odisha', 'orissa'])][['NAME_2', 'geometry']].rename(columns={'NAME_2': 'district_src'})
        dist_src = 'GADM 4.1'
    except Exception as e:
        print('  ! GADM read failed:', e)
if districts is None:
    try:
        meta = requests.get('https://www.geoboundaries.org/api/current/gbOpen/IND/ADM2/', timeout=60).json()
        gb = download(meta['gjDownloadURL'], DIRS['raw'] / 'geoBoundaries-IND-ADM2.geojson', timeout=300)
        g = gpd.read_file(gb)
        g = g[g.representative_point().within(box(*BBOX_ODISHA))]
        districts = g[['shapeName', 'geometry']].rename(columns={'shapeName': 'district_src'})
        dist_src = 'geoBoundaries ADM2 (bbox filter; check edges)'
    except Exception as e:
        print('  ! geoBoundaries failed:', e)

if districts is not None:
    districts = districts.to_crs(CRS_GEO)
    alias = {a: k for k, v in STUDY_DISTRICTS.items() for a in v}
    districts['district'] = districts.district_src.map(lambda s: alias.get(norm(s), str(s)))
    districts['study'] = districts.district.isin(STUDY_DISTRICTS)
    missing = set(STUDY_DISTRICTS) - set(districts.district)
    print(f'{dist_src}: {len(districts)} districts; study districts matched: {districts.study.sum()}/8',
          f'| NOT matched: {missing}' if missing else '')
    j = gpd.sjoin(gdf_towers[['tower_id', 'geometry']], districts[['district', 'study', 'geometry']], how='left', predicate='within')
    j = j.drop_duplicates('tower_id').set_index('tower_id')
    gdf_towers['district'] = gdf_towers.tower_id.map(j.district)
    gdf_towers['study_district'] = gdf_towers.tower_id.map(j.study).fillna(False).astype(bool)
    log_data('District boundaries', dist_src, gadm_zip or 'geoBoundaries', len(districts))
else:
    print('  ! No district layer: the study domain will use distance to coast only')
    gdf_towers['district'] = None
    gdf_towers['study_district'] = True
    log_data('District boundaries', 'GADM / geoBoundaries', None, None, 'failed')

gdf_towers['in_study_domain'] = gdf_towers.study_district & (gdf_towers.dist_coast_km <= 60)
print(f'Towers in study domain (study districts, ≤60 km from coast): {gdf_towers.in_study_domain.sum():,}')

## 5. STORM synthetic tracks (North Indian basin, present climate)
The STORM files are comma-separated text with 13 columns (Bloemendaal et al. 2020):
`year, month, tc_number, time_step(3-h), basin_id, lat, lon, min_pressure(hPa), max_wind(m/s, 10-min), rmw(km), category, landfall, dist_to_land(km)`.

The 4TU download is a zip that may contain further zips, so the code looks inside recursively and extracts only the North Indian (`_NI_`) present-climate files.

In [ ]:
STORM_COLS = ['year', 'month', 'tc_num', 'step', 'basin', 'lat', 'lon', 'pres_hpa', 'vmax10_ms', 'rmw_km', 'cat', 'landfall', 'dist_land_km']
STORM_DIR = DIRS['interim'] / 'storm_NI'
STORM_DIR.mkdir(exist_ok=True)
NI_PAT = re.compile(r'STORM_DATA_IBTRACS_NI_1000_YEARS_(\d+)\.(txt|csv)$', re.I)

def extract_storm_ni(zpath, out_dir, depth=0):
    """Extract STORM NI files from a zip, looking inside nested zips."""
    found = []
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            base = Path(info.filename).name
            if NI_PAT.search(base):
                target = out_dir / base
                if not target.exists():
                    with z.open(info) as s, open(target, 'wb') as t:
                        shutil.copyfileobj(s, t)
                found.append(target)
            elif base.lower().endswith('.zip') and depth < 3 and ('NI' in base.upper() or 'IBTRACS' in base.upper() or depth == 0):
                inner = out_dir / ('_inner_' + base)
                with z.open(info) as s, open(inner, 'wb') as t:
                    shutil.copyfileobj(s, t)
                found += extract_storm_ni(inner, out_dir, depth + 1)
                inner.unlink(missing_ok=True)
    return found

def fetch_storm_from_4tu(article_id=12706085):
    """Download the North Indian STORM files straight from 4TU.ResearchData (free), so no Drive upload is needed."""
    try:
        files = requests.get(f'https://data.4tu.nl/v2/articles/{article_id}/files', timeout=60).json()
    except Exception as e:
        print('  ! could not list 4TU files:', e)
        return []
    print('Files in the 4TU record:', [f.get('name') for f in files])
    want = [f for f in files if NI_PAT.search(f.get('name', '')) or
            (re.search(r'(^|[_\W])NI([_\W]|$)', f.get('name', '')) and f.get('name', '').lower().endswith('.zip'))]
    if not want:   # fall back to the archive of everything
        want = [f for f in files if f.get('name', '').lower().endswith('.zip')]
    got = []
    for f in want:
        print(f"  downloading {f['name']} ({f.get('size', 0) / 1e6:.0f} MB) …")
        p = download(f['download_url'], DIRS['raw'] / f['name'], timeout=1800)
        if p:
            got.append(p)
    return got

storm_files = sorted(STORM_DIR.glob('STORM_DATA_IBTRACS_NI_1000_YEARS_*'))
if len(storm_files) < 10:
    zips = [p for p in DIRS['raw'].glob('*.zip') if 'storm' in p.name.lower()]
    if not zips:
        print(f'No STORM zip in {DIRS["raw"]}; trying a direct download from 4TU …')
        got = fetch_storm_from_4tu()
        for p in got:
            if NI_PAT.search(p.name):
                shutil.copy(p, STORM_DIR / p.name)
        zips = [p for p in got if p.suffix.lower() == '.zip']
    if not zips and len(sorted(STORM_DIR.glob('STORM_DATA_IBTRACS_NI_1000_YEARS_*'))) < 10:
        raise FileNotFoundError(f'STORM not found. Upload the STORM zip to {DIRS["raw"]} (or check the 4TU file list above).')
    print('Extracting from:', [z.name for z in zips])
    for zp in zips:
        extract_storm_ni(zp, STORM_DIR)
    storm_files = sorted(STORM_DIR.glob('STORM_DATA_IBTRACS_NI_1000_YEARS_*'))
print(f'{len(storm_files)} STORM NI files:', [f.name for f in storm_files])
if len(storm_files) != 10:
    print('  ! Expected 10 files (10 × 1,000 years). Check the zip contents.')

In [ ]:
parts = []
for f in storm_files:
    k = int(NI_PAT.search(f.name).group(1))
    d = pd.read_csv(f, header=None, names=STORM_COLS)
    d['file_k'] = k
    parts.append(d)
storm = pd.concat(parts, ignore_index=True)
for c in ['year', 'month', 'tc_num', 'step', 'basin', 'cat', 'landfall', 'file_k']:
    storm[c] = storm[c].astype(int)
storm['sim_year'] = storm.file_k * 1000 + storm.year            # 0 … 9,999
storm['storm_id'] = storm.sim_year.astype(str).str.zfill(4) + '_' + storm.tc_num.astype(str).str.zfill(2)
storm.loc[storm.lon > 180, 'lon'] -= 360
N_YEARS = 1000 * storm.file_k.nunique()

# Checks on the column layout
print(f'Rows {len(storm):,} | storms {storm.storm_id.nunique():,} | simulated years {N_YEARS:,}')
print('basin ids:', storm.basin.unique(), '(NI = 2)')
print(storm[['lat', 'lon', 'pres_hpa', 'vmax10_ms', 'rmw_km', 'dist_land_km']].describe().loc[['min', '50%', 'max']].round(1))
assert storm.lat.between(-5, 35).all() and storm.vmax10_ms.between(0, 120).all(), 'Column layout looks wrong'
print(f'Genesis rate: {storm.storm_id.nunique() / N_YEARS:.2f} storms per year (Bloemendaal 2020a: λ≈2.0 for NI)')

In [ ]:
# Keep storms that pass within TRACK_BUFFER_KM of the Odisha coast (minimum distance from any track point to the coastline)
R_E = 6371.0
def to_xyz(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return np.c_[np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]

odisha_coast = gpd.clip(coast, box(84.6, 19.0, 87.6, 21.8)).to_crs(CRS_M)
dens = union(odisha_coast).segmentize(2000)                    # vertex every ≤2 km
cxy = gpd.GeoSeries(gpd.points_from_xy(*np.array(
    [c for g in getattr(dens, 'geoms', [dens]) for c in g.coords]).T), crs=CRS_M).to_crs(CRS_GEO)
tree = cKDTree(to_xyz(cxy.y.values, cxy.x.values))
chord, _ = tree.query(to_xyz(storm.lat.values, storm.lon.values))
storm['dist_odisha_coast_km'] = 2 * R_E * np.arcsin(np.clip(chord / 2, 0, 1))

closest = storm.loc[storm.groupby('storm_id').dist_odisha_coast_km.idxmin()]
keep_ids = closest.loc[closest.dist_odisha_coast_km <= TRACK_BUFFER_KM, 'storm_id']
storm_od = storm[storm.storm_id.isin(keep_ids)].copy()
storm_od.to_parquet(DIRS['processed'] / 'storm_NI_odisha300km.parquet', index=False)

cl = closest[closest.storm_id.isin(keep_ids)]
print(f'Storms within {TRACK_BUFFER_KM} km of the Odisha coast: {len(keep_ids):,} → {len(keep_ids) / N_YEARS:.2f} per year')
for thr, lab in [(33, '≥33 m/s (10-min, ~cyclonic storm and above)'), (43, '≥43 m/s'), (50, '≥50 m/s')]:
    n = (cl.vmax10_ms >= thr).sum()
    print(f'  at closest approach {lab}: {n:,} storms → mean return period {N_YEARS / max(n, 1):.1f} yr')
log_data('STORM present-climate tracks, NI basin', 'https://data.4tu.nl/articles/_/12706085', STORM_DIR, len(storm))
log_data('STORM tracks within 300 km of Odisha coast', 'derived', DIRS['processed'] / 'storm_NI_odisha300km.parquet', len(storm_od))

## 6. IBTrACS (North Indian basin): Fani and validation cyclones
IBTrACS v04r01 CSV from NOAA NCEI. Wind fields: `NEWDELHI_WIND` (IMD, 3-min, kt), `USA_WIND` (JTWC, 1-min, kt), `USA_RMW` (nmi).

In [ ]:
IBT_URL = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NI.list.v04r01.csv'
ibt_csv = download(IBT_URL, DIRS['raw'] / 'ibtracs.NI.list.v04r01.csv', timeout=300)
ibt = None
if ibt_csv:
    ibt = pd.read_csv(ibt_csv, skiprows=[1], low_memory=False, na_values=[' ', ''])
    keep = ['SID', 'SEASON', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES', 'DIST2LAND', 'LANDFALL',
            'NEWDELHI_GRADE', 'NEWDELHI_WIND', 'NEWDELHI_PRES', 'NEWDELHI_CI', 'NEWDELHI_DP', 'NEWDELHI_POCI',
            'USA_WIND', 'USA_PRES', 'USA_RMW', 'USA_ROCI', 'USA_POCI', 'STORM_SPEED', 'STORM_DIR']
    ibt = ibt[[c for c in keep if c in ibt.columns]].copy()
    ibt['ISO_TIME'] = pd.to_datetime(ibt.ISO_TIME)
    for c in ibt.columns.difference(['SID', 'NAME', 'NATURE', 'ISO_TIME', 'NEWDELHI_GRADE']):
        ibt[c] = pd.to_numeric(ibt[c], errors='coerce')
    chord, _ = tree.query(to_xyz(ibt.LAT.values, ibt.LON.values))
    ibt['dist_odisha_coast_km'] = 2 * R_E * np.arcsin(np.clip(chord / 2, 0, 1))
    print(f'IBTrACS NI: {ibt.SID.nunique():,} storms, {ibt.SEASON.min()}–{ibt.SEASON.max()}')
    log_data('IBTrACS v04r01 NI', IBT_URL, ibt_csv, len(ibt))
else:
    log_data('IBTrACS v04r01 NI', IBT_URL, None, None, 'failed')

In [ ]:
EVENTS = {'FANI_2019': ('FANI', 2019), 'PHAILIN_2013': ('PHAILIN', 2013), 'TITLI_2018': ('TITLI', 2018),
          'HUDHUD_2014': ('HUDHUD', 2014), 'BULBUL_2019': ('BULBUL', 2019), 'AMPHAN_2020': ('AMPHAN', 2020),
          'YAAS_2021': ('YAAS', 2021), 'DANA_2024': ('DANA', 2024)}
events = {}
if ibt is not None:
    for key, (nm, yr) in EVENTS.items():
        e = ibt[(ibt.NAME.str.upper() == nm) & (ibt.SEASON == yr)]
        if len(e):
            events[key] = e
    # 1999 Odisha super cyclone: unnamed in most archives, so pick the strongest Oct–Nov 1999 storm passing within 100 km of the coast
    c99 = ibt[(ibt.SEASON == 1999) & ibt.ISO_TIME.dt.month.isin([10, 11])]
    near = c99.groupby('SID').dist_odisha_coast_km.min()
    near = near[near < 100].index
    if len(near):
        vmax = c99[c99.SID.isin(near)].groupby('SID')[['USA_WIND', 'NEWDELHI_WIND']].max().max(axis=1)
        events['SUPERCYCLONE_1999'] = c99[c99.SID == vmax.idxmax()]
    rows = []
    for key, e in events.items():
        e.to_csv(DIRS['processed'] / f'ibtracs_{key}.csv', index=False)
        near_pt = e.loc[e.dist_odisha_coast_km.idxmin()]
        rows.append(dict(event=key, sid=e.SID.iloc[0], start=e.ISO_TIME.min(), end=e.ISO_TIME.max(),
                         max_imd_kt=e.NEWDELHI_WIND.max(), max_jtwc_kt=e.USA_WIND.max(), min_pres=e.WMO_PRES.min(),
                         closest_km=round(near_pt.dist_odisha_coast_km, 1), closest_time=near_pt.ISO_TIME,
                         lat=near_pt.LAT, lon=near_pt.LON, rmw_nmi_at_closest=near_pt.get('USA_RMW')))
    ev_table = pd.DataFrame(rows).sort_values('start')
    ev_table.to_csv(DIRS['tables'] / 'validation_events.csv', index=False)
    print(ev_table.to_string(index=False))

    # Empirical rate of storms passing within 300 km of the Odisha coast (1980 onwards, satellite era), as a check on STORM
    sat = ibt[ibt.SEASON >= 1980]
    mind = sat.groupby('SID').dist_odisha_coast_km.min()
    n_years = sat.SEASON.max() - 1980 + 1
    print(f'IBTrACS 1980–{sat.SEASON.max()}: {(mind <= TRACK_BUFFER_KM).sum()} storms within {TRACK_BUFFER_KM} km '
          f'→ {(mind <= TRACK_BUFFER_KM).sum() / n_years:.2f}/yr (STORM: {len(keep_ids) / N_YEARS:.2f}/yr; '
          'IBTrACS also counts depressions, so compare after filtering by intensity in Notebook 02)')

# Fani track: IBTrACS if available, else the IMD 3-hourly track in the Raj repo
if 'FANI_2019' in events:
    fani = events['FANI_2019'][['ISO_TIME', 'LAT', 'LON', 'NEWDELHI_WIND', 'USA_WIND', 'USA_RMW']].rename(columns=str.lower)
    lf = fani.loc[(fani.lat - FANI_LANDFALL[1]).abs().add((fani.lon - FANI_LANDFALL[0]).abs()).idxmin()]
    print('Nearest IBTrACS fix to the assumed landfall:', lf.to_dict())
else:
    f_imd = pd.read_excel(REPO_DIR / 'Step1 - WS_estimation_WDEmodel_fragility' / 'FANI.xlsx')
    fani = f_imd.rename(columns={'Lat': 'lat', 'Long': 'lon', 'Vmax (kts)': 'newdelhi_wind'})
    print('Using the IMD 3-hourly Fani track from the Raj repo (IBTrACS not available).')
fani.to_csv(DIRS['processed'] / 'fani_track.csv', index=False)

## 7. Station winds during Fani (NOAA ISD via Meteostat, free)
Hourly mean wind (`wspd`) and peak gust (`wpgt`) in km/h at stations on and near the Odisha coast, 25 Apr – 6 May 2019. Synoptic means are about 10-minute averages, and gust reporting is patchy. These are used only to check the modelled wind field in Notebook 01.

In [ ]:
try:
    from meteostat import Stations, Hourly
    st = Stations().bounds((22.6, 84.0), (18.5, 87.6)).fetch()
    print(f'{len(st)} stations in the box:', ', '.join(st.name.astype(str).head(20)))
    obs = Hourly(list(st.index), datetime(2019, 4, 25), datetime(2019, 5, 6, 23)).fetch()
    obs = obs.reset_index().merge(st[['name', 'latitude', 'longitude', 'elevation']], left_on='station', right_index=True)
    obs.to_csv(DIRS['processed'] / 'fani_station_obs_meteostat.csv', index=False)
    peak = (obs.groupby(['station', 'name']).agg(max_wspd_kmh=('wspd', 'max'), max_wpgt_kmh=('wpgt', 'max'),
                                                 n_hours=('wspd', 'count')).sort_values('max_wspd_kmh', ascending=False))
    print(peak.head(15).to_string())
    log_data('Station observations, Fani (Meteostat / NOAA ISD)', 'meteostat.net', DIRS['processed'] / 'fani_station_obs_meteostat.csv', len(obs))
except Exception as e:
    print('  ! Station data step skipped:', type(e).__name__, e)
    log_data('Station observations, Fani (Meteostat / NOAA ISD)', 'meteostat.net', None, None, 'failed', str(e)[:120])

## 8. Save the processed layers

In [ ]:
tower_cols = ['tower_id', 'lat', 'lon', 'line_id', 'line_name', 'voltage_kv', 'from_sub', 'to_sub', 'span_m',
              'dist_coast_km', 'band', 'district', 'study_district', 'in_study_domain', 'dist_fani_landfall_km',
              'in_fani_zone', 'damaged', 'collapsed', 'damage_state', 'optcl_line_name', 'damage_observed', 'geometry']
gdf_towers = gdf_towers[tower_cols]
gpkg = DIRS['processed'] / 'odisha_network.gpkg'
gdf_towers.to_file(gpkg, layer='towers', driver='GPKG')
gdf_lines.to_file(gpkg, layer='lines', driver='GPKG')
gdf_subs.to_file(gpkg, layer='substations', driver='GPKG')
coast.to_file(gpkg, layer='coastline', driver='GPKG')
if districts is not None:
    districts.to_file(gpkg, layer='districts', driver='GPKG')
gdf_towers.drop(columns='geometry').to_parquet(DIRS['processed'] / 'towers.parquet', index=False)
gdf_towers.drop(columns='geometry').to_csv(DIRS['processed'] / 'towers.csv', index=False)
print('Saved', gpkg, '(open it in QGIS: layers towers, lines, substations, coastline, districts)')

## 9. Summary tables and QA map

In [ ]:
t1 = pd.crosstab(gdf_towers.band, gdf_towers.voltage_kv, margins=True).reindex([b[2] for b in BANDS] + ['All'])
t1.to_csv(DIRS['tables'] / 'towers_by_band_voltage.csv')
print('Towers by coastal band and voltage (kV):\n', t1.to_string())

t2 = (gdf_towers[gdf_towers.in_study_domain].fillna({'district': 'unassigned'}).groupby(['district', 'band']).agg(
        towers=('tower_id', 'size'), damaged_fani=('damaged', 'sum'), collapsed_fani=('collapsed', 'sum'),
        median_span_m=('span_m', 'median')).round(0))
t2.to_csv(DIRS['tables'] / 'study_domain_by_district_band.csv')
print('\nStudy domain by district and band:\n', t2.to_string())

t3 = (gdf_towers[gdf_towers.damaged].groupby(['optcl_line_name', 'voltage_kv', 'damage_state']).size()
        .unstack(fill_value=0))
t3['total'] = t3.sum(axis=1)
t3 = t3.sort_values('total', ascending=False)
t3.to_csv(DIRS['tables'] / 'fani_damage_by_line.csv')
print('\nFani damage by line (candidate pilot corridors):\n', t3.to_string())

# CDRI span indicator: 400 kV > 400 m, 220/132 kV > 250 m
lim = gdf_towers.voltage_kv.map({400: 400, 765: 400, 220: 250, 132: 250})
gdf_towers['long_span_flag'] = gdf_towers.span_m > lim
print('\nShare of spans above the CDRI threshold, study domain:')
print(gdf_towers[gdf_towers.in_study_domain].groupby('voltage_kv').long_span_flag.mean().round(2).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 9))
for a, (xlim, ylim, title) in zip(ax, [((81.3, 87.7), (17.7, 22.7), 'Odisha transmission network and coastal bands'),
                                        ((85.0, 86.3), (19.6, 20.6), 'Fani zone: damaged towers (OPTCL / Raj et al.)')]):
    if districts is not None:
        districts.boundary.plot(ax=a, color='#bbbbbb', lw=0.5)
        districts[districts.study].boundary.plot(ax=a, color='#555555', lw=1.0)
    for d_km, ls in [(20, '--'), (60, ':')]:
        gpd.GeoSeries([coast_m.buffer(d_km * 1000)], crs=CRS_M).to_crs(CRS_GEO).boundary.plot(ax=a, color='#2b8cbe', lw=0.8, ls=ls)
    coast.plot(ax=a, color='#08519c', lw=1.2)
    for kv, col in [(132, '#fdae61'), (220, '#f46d43'), (400, '#a50026'), (765, '#4d004b')]:
        gdf_lines[gdf_lines.voltage_kv == kv].plot(ax=a, color=col, lw=0.8, label=f'{kv} kV')
    gdf_subs.plot(ax=a, color='k', markersize=6, zorder=4)
    dm = gdf_towers[gdf_towers.damaged]
    dm[dm.collapsed].plot(ax=a, color='red', marker='x', markersize=25, zorder=5, label='Fani: collapsed')
    dm[~dm.collapsed].plot(ax=a, color='orange', marker='^', markersize=18, zorder=5, label='Fani: partial')
    if {'lat', 'lon'} <= set(fani.columns):
        a.plot(fani.lon, fani.lat, '-o', color='purple', ms=2, lw=1, label='Fani track')
    a.set_xlim(xlim); a.set_ylim(ylim); a.set_aspect('equal'); a.set_title(title)
ax[0].legend(loc='lower right', fontsize=8)
ax[0].text(0.01, 0.01, 'Coastal bands: 20 km (dashed), 60 km (dotted)', transform=ax[0].transAxes, fontsize=8)
plt.tight_layout()
fig.savefig(DIRS['figures'] / '00_network_qa_map.png', dpi=200)
plt.show()

In [ ]:
inv = pd.DataFrame(INVENTORY)
inv.to_csv(DIRS['tables'] / 'data_inventory.csv', index=False)
print(inv[['dataset', 'status', 'records', 'local_path']].to_string())
print('\nDone. Next: Notebook 01 (Fani wind field and tower damage diagnostic).')